# Exploratory Data Analysis — Superstore Dataset

## Pendahuluan

Sumber dataset yang digunakan berasal dari Kaggle dengan judul "Superstore Dataset" ([Link dataset](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)).

Notebook ini berisi **EDA lengkap**. Untuk pemodelan *data mining* (Clustering & Association Rule Mining) beserta EDA ringkasnya, lihat `Model.ipynb`.

**Fitur-Fitur**

| Fitur | Deskripsi |
|---|---|
| Row ID | Unique ID for each row. |
| Order ID | Unique Order ID for each Customer. |
| Order Date | Order Date of the product. |
| Ship Date | Shipping Date of the Product. |
| Ship Mode | Shipping Mode specified by the Customer. |
| Customer ID | Unique ID to identify each Customer. |
| Customer Name | Name of the Customer. |
| Segment | The segment where the Customer belongs. |
| Country | Country of residence of the Customer. |
| City | City of residence of the Customer. |
| State | State of residence of the Customer. |
| Postal Code | Postal Code of every Customer. |
| Region | Region where the Customer belong. |
| Product ID | Unique ID of the Product. |
| Category | Category of the product ordered. |
| Sub-Category | Sub-Category of the product ordered. |
| Product Name | Name of the Product. |
| Sales | Sales of the Product. |
| Quantity | Quantity of the Product. |
| Discount | Discount provided. |
| Profit | Profit/Loss incurred. |

## Identifikasi dan Ekstraksi Data

### 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # muat plotly.js dari CDN (notebook tetap ringan)

sns.set_theme(style="whitegrid")

### 2. Load Data

Load data `Superstore.csv`.

In [ ]:
df = pd.read_csv('Superstore.csv', encoding="latin1")

### 3. Data Examination

Tampilkan 5 baris pertama untuk melihat gambaran data.

In [ ]:
df.head()

Melihat tipe data yang dimiliki untuk tiap-tiap kolom.

In [ ]:
df.info()

Melihat ukuran dataset.

In [ ]:
df.shape

Dataset terdiri dari 9994 baris dan 21 kolom.

### 4. Check Duplicate Rows

In [ ]:
df.duplicated().sum()

In [ ]:
df.duplicated(subset=["Order ID", "Product ID", "Order Date", "Ship Date"]).sum()

In [ ]:
df[df.duplicated(subset=["Order ID", "Product ID", "Order Date"], keep=False)]

Terdapat 8 transaksi yang memiliki `Product ID` lebih dari satu dalam satu `Order ID`, namun dengan `Quantity` yang berbeda. Oleh karena itu, transaksi tersebut akan dihapus.

### 5. Summary Statistics

Melihat rangkuman statistika untuk tiap kolom dengan tujuan mempermudah dalam memahami data serta persebaran data.

In [ ]:
df.describe(include='all')

### 6. Identify Missing Values

In [ ]:
df.isnull().sum()

Tidak ada *missing values* pada dataset ini.

## Transformasi dan Pembersihan Data

### 1. Remove Duplicate Values

In [ ]:
print("Sebelum duplikat dihapus: ", df.duplicated(subset=["Order ID", "Product ID", "Order Date", "Ship Date"]).sum())
df = df.drop_duplicates(subset=["Order ID", "Product ID", "Order Date"], keep=False)
print("Setelah duplikat dihapus: ", df.duplicated(subset=["Order ID", "Product ID", "Order Date", "Ship Date"]).sum())

### 2. Ubah Tipe Data Kolom Date dari object ke datetime

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])
df.info()

### 3. Hilangkan Kolom yang Tidak Diperlukan

In [ ]:
df.drop(columns=["Row ID"], inplace=True)
df.head()

## Exploratory Data Analysis

### 1. Univariate Analysis

#### Order

In [ ]:
# Jumlah transaksi
df["Order ID"].nunique()

Terdapat 5007 transaksi.

#### Region and State

In [ ]:
# Cek unique value dari Country
df["Country"].unique()

Hanya terdapat satu negara, yaitu United States.

In [ ]:
df["State"].value_counts()

In [ ]:
us_state_to_abbrev = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS", "Missouri": "MO",
    "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Virginia": "VA", "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
    "District of Columbia": "DC",
}

In [ ]:
# Buat kolom baru berisi singkatan State
df['State_abb'] = df['State'].replace(us_state_to_abbrev)

fig = go.Figure(data=go.Choropleth(
    locations=df['State_abb'].value_counts().index,
    z=df['State_abb'].value_counts(),
    locationmode='USA-states',
    colorscale='teal', zmin=1, zmax=1000,
))
fig.update_layout(
    font=dict(size=14),
    title={'text': "Number of Customers by State", 'y': 0.95, 'x': 0.5},
    geo_scope='usa',
)
fig.show()

#### Ship Mode

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='Ship Mode', order=df['Ship Mode'].value_counts().index)
plt.title('Order Count by Ship Mode')
plt.xticks(rotation=90)
plt.xlabel('Ship Mode')
plt.ylabel('Order Count')
plt.show()

#### Customer

In [ ]:
df['Customer ID'].nunique()

Terdapat 793 *customer* yang melakukan transaksi.

#### Customer Segment

In [ ]:
segment_counts = df['Segment'].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Order Count by Segment')
plt.axis('equal')
plt.show()

#### Product

In [ ]:
# Cek jumlah unique value dari Product ID
df["Product ID"].nunique()

Terdapat 1862 produk yang berbeda.

In [ ]:
top_10_products = df['Product Name'].value_counts().nlargest(10)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_10_products.index, y=top_10_products.values)
plt.title('Top 10 Products by Order Count')
plt.xticks(rotation=90)
plt.xlabel('Product Name')
plt.ylabel('Order Count')
plt.show()

In [ ]:
top_10_by_quantity = df.groupby('Product Name')['Quantity'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_10_by_quantity.index, y=top_10_by_quantity.values)
plt.title('Top 10 Products by Quantity Ordered')
plt.xlabel('Product Name')
plt.ylabel('Total Quantity')
plt.xticks(rotation=90)
plt.show()

#### Product Category

In [ ]:
category_counts = df['Category'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(category_counts, labels=category_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Order Count by Category')
plt.axis('equal')
plt.show()

#### Sub-Category

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='Sub-Category', order=df['Sub-Category'].value_counts().index)
plt.title('Order Count by Sub-Category')
plt.xticks(rotation=90)
plt.xlabel('Sub-Category')
plt.ylabel('Order Count')
plt.show()

### 2. Bivariate Analysis

#### Korelasi

In [ ]:
numeric_df = df.select_dtypes(include=['int32', 'int64', 'float64'])
if 'Postal Code' in numeric_df.columns:
    numeric_df = numeric_df.drop(columns=['Postal Code'])

plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True)
plt.title('Korelasi Kolom Numerik')
plt.xticks(rotation=30)
plt.show()

Sales memiliki korelasi positif terhadap Profit, yaitu sebesar 0.48. Sementara itu, Discount memiliki korelasi negatif terhadap Profit, yaitu sebesar -0.22. Hal ini menunjukkan bahwa semakin besar diskon yang diberikan, maka profit yang didapatkan cenderung semakin kecil.

#### Sales vs Profit

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='Sales', y='Profit', alpha=0.7)
plt.title('Sales vs Profit')
plt.xlabel('Sales')
plt.ylabel('Profit')
plt.show()

#### Total Profit vs Category

In [ ]:
profit_summary = df.groupby('Category')['Profit'].sum().reset_index()
profit_summary = profit_summary.sort_values(by='Profit', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=profit_summary, x='Category', y='Profit')
plt.title('Total Profit by Category')
plt.xlabel('Category')
plt.ylabel('Total Profit')
plt.xticks(rotation=45)
plt.show()

#### Total Profit vs Sub-Category

In [ ]:
profit_summary = df.groupby('Sub-Category')['Profit'].sum().reset_index()
profit_summary = profit_summary.sort_values(by='Profit', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=profit_summary, x='Sub-Category', y='Profit')
plt.title('Total Profit by Sub-Category')
plt.xlabel('Sub-Category')
plt.ylabel('Total Profit')
plt.xticks(rotation=45)
plt.show()

## Kesimpulan EDA

- Dataset mencakup **5007 transaksi** dari **793 customer** di **satu negara (United States)**, dengan 1862 produk unik.
- Segmen **Consumer** mendominasi jumlah order, diikuti Corporate dan Home Office.
- Kategori **Office Supplies** paling sering dipesan; **Binders** dan **Paper** adalah sub-kategori dengan order terbanyak.
- **Sales** berkorelasi positif dengan **Profit** (0.48), sedangkan **Discount** berkorelasi negatif dengan **Profit** (-0.22).
- Sub-kategori **Tables**, **Bookcases**, dan **Supplies** membukukan **profit negatif**, sementara **Copiers**, **Phones**, dan **Accessories** paling menguntungkan.